# Disaster response triage: train DistilBERT on a GPU

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. Training (2 epochs on 53,531 tweets) takes about 10–20 minutes on a T4.
3. The last cell downloads `distilbert-humaid.zip` and `results.zip`. Unzip both inside the project folder on your laptop (`C:\Users\BIT\disaster-response-triage`), so you get `models/distilbert-humaid/` and an updated `results/`.

Colab storage is wiped when the session ends, so download the zips before closing the tab.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Get the code
The repository is private, so paste a GitHub token with access to it (GitHub → Settings → Developer settings → Personal access tokens). The input is hidden and the token is removed from the clone afterwards.

In [ ]:
import os, subprocess
from getpass import getpass

REPO = "github.com/ayush123-code/disaster-response-triage.git"
DEST = "/content/disaster-response-triage"

if not os.path.exists(DEST):
    token = getpass("GitHub token (hidden): ").strip()
    result = subprocess.run(["git", "clone", "-q", f"https://{token}@{REPO}", DEST], capture_output=True, text=True)
    del token
    if result.returncode != 0:
        raise RuntimeError("Clone failed: check that the token is valid and has access to the repository.")
    subprocess.run(["git", "-C", DEST, "remote", "set-url", "origin", f"https://{REPO}"], check=True)

%cd {DEST}
!git log --oneline -3

In [ ]:
!pip install -q -U transformers accelerate
!python download_data.py

## Train
Keeps the checkpoint with the best validation macro F1, then evaluates it on the 15,160-tweet HumAID test set.

In [ ]:
!python train_distilbert.py --epochs 2 --save-every 1000

## Score the Kerala floods replay with the new model

In [ ]:
!python score_kerala.py > /dev/null

import json
from IPython.display import Image, display

for f in ["results/baseline_metrics.json", "results/distilbert_metrics.json"]:
    m = json.load(open(f))
    print(f"{m['model']:32s} accuracy {m['accuracy']:.3f}   macro F1 {m['macro_f1']:.3f}")

r = json.load(open("results/ranking.json"))
print(f"\nKerala replay ({r['category_model']}): {r['posts']} posts, {r['posts_needing_help']} need help")
print("Priority counts:", r["priority_counts"])
for name, v in r["ranking"].items():
    print(f"{name:28s} top-50 precision {v['precision_at_50']:.0%}   posts read to find 90%: {v['posts_read_for_90pct']}")

display(Image("results/distilbert_confusion.png", width=650))
display(Image("results/ranking_curve.png", width=650))

## Download the model and results

In [ ]:
!zip -qr distilbert-humaid.zip models/distilbert-humaid
!zip -qr results.zip results

from google.colab import files
files.download("distilbert-humaid.zip")
files.download("results.zip")